# Experimento A: lógica del agente en simulación
Simula una partida completa guiada por cada política (reglas, ID3, Q-Learning) con un jugador simulado. Ejecutar desde `codigo/ia_models` (kernel con `requirements.txt`).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()))
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()))
import numpy as np
from pictomarket_ia.datos import load_data
from pictomarket_ia.cargar import load_policy
from pictomarket_ia.caracteristicas import features
from pictomarket_ia.juego import ACTION_NAME, choose_product, start_game
from pictomarket_ia.usuario_sim import PROFILES, SimulatedLearner
data = load_data()

In [ ]:
def jugar(policy_name, profile='apoyo', scenario=1, seed=7, verbose=True):
    policy = load_policy(policy_name)
    rng = np.random.default_rng(seed)
    ch = data['retos'][scenario]
    order = ch['lista_correcta'] + ch['distractores']
    rng.shuffle(order)
    state = start_game(data, scenario, [int(x) for x in order])
    learner = SimulatedLearner(PROFILES[profile], np.random.default_rng(seed + 1))
    nombre = lambda p: data['catalogo'][str(p)]['nombre']
    turn = now = last = 0
    while state.items_restantes and turn < 40:
        turn += 1
        product, t = learner.touch(data, state, last)
        now += t
        feats = features(data, state)
        wrong = product not in state.items_restantes
        action = policy.act(feats) if wrong else 0
        state = choose_product(data, state, product, now, help_action=lambda _s, a=action: a)
        last = state.log[-1].action
        if verbose:
            print(f'T{turn}: busca {nombre(state.log[-1].target)} | toca {nombre(product)} | s={feats} -> {ACTION_NAME[action]}')
    return turn, state.errores_totales

In [ ]:
for name in ('reglas', 'id3', 'qlearning'):
    print('==', name)
    print(jugar(name))

Las métricas completas (30 semillas) están en `datos/simulaciones/metricas_resumen.csv` y se regeneran con `python experimentos/entrenar_y_evaluar.py`.